In [71]:
import pandas as pd
import re


In [72]:
Path = r'E:\_Projects\RFC\\'

In [73]:
#data = pd.read_csv(Path+'Input\\Logic.csv')
data_pivot = pd.read_csv(Path+'Input\\Pivot.csv')

data_pivot_Bk = data_pivot.copy()

data_pivot['Class'] = data_pivot['BOM Parent Product ID'].str[0:4]

In [74]:
data = pd.read_excel(Path+'Input\\Logic.xlsx', sheet_name='PPL')

### Replace , and within () to OR

In [75]:

def replace_commas(match):
    return match.group(0).replace(',', ' , ')


def replace_commas_to_or(match):
    return match.group(0).replace(',', ' OR')


data['Function'] = data['Function'].apply(lambda x: re.sub(r'\(.*?\)', replace_commas, x)if isinstance(x, str) else x)
data['Function'] = data['Function'].apply(lambda x: re.sub(r'\(.*?\)', replace_commas_to_or, x)if isinstance(x, str) else x)




In [76]:
data_pv = data_pivot[['Class', 'BOM Parent Product ID']]
data_pv = data_pv.copy()

df = data.copy()
df = df[(df['Function'].notna()) & (df['Function'].str.strip() != '')]
df['Class'] = df['Item'].str[0:4]

df_bk = df.copy()

## Feature Extraction

In [77]:
import pandas as pd
import re
def extract_conditions(text):
   if pd.isna(text) or text == '':
       return {'Fun_Or': '', 'Fun_And': ''}
   result = {}
   remaining_text = text
   # First process parenthesized OR groups
   or_groups = re.findall(r'\((.*?)\b(?:OR|or)\b(.*?)\)', text, re.IGNORECASE)
   # Process each OR group
   for i, group in enumerate(or_groups):
       or_part = ' '.join([part.strip() for part in group if part.strip()])
       codes = re.findall(r'\b(F\d{3}|OP\d{2}|D\d{3}|PS\d{2}|PS\d{3}|VR\d{2}|VF\d{2})\b', or_part)
       col_name = 'Fun_Or' if i == 0 else f'Fun_Or_{i}'
       result[col_name] = f"[{', '.join(codes)}]" if codes else ''
       remaining_text = remaining_text.replace(f"({or_part})", "", 1)
   # Now process non-parenthesized OR conditions in remaining text
   or_segments = [seg.strip() for seg in remaining_text.split(',')
                 if seg.strip() and re.search(r'\b(?:OR|or)\b', seg, re.IGNORECASE)]
   if or_segments and 'Fun_Or' not in result:
       # Only process if we haven't found any parenthesized OR groups yet
       all_or_codes = []
       for segment in or_segments:
           codes = re.findall(r'\b(F\d{3}|OP\d{2}|D\d{3}|PS\d{2}|PS\d{3}|VR\d{2}|VF\d{2})\b', segment)
           all_or_codes.extend(codes)
       if all_or_codes:
           result['Fun_Or'] = f"[{', '.join(all_or_codes)}]"
   # Process AND conditions in remaining text
   and_conditions = []
   segments = [seg.strip() for seg in remaining_text.split(',') if seg.strip()]
   for segment in segments:
       if not re.search(r'\b(?:OR|or)\b', segment, re.IGNORECASE):
           if code := re.search(r'\b(F\d{3}|OP\d{2}|D\d{3}|PS\d{2}|PS\d{3}|VR\d{2}|VF\d{2})\b', segment):
               and_conditions.append(code.group())
   and_conditions = list(dict.fromkeys(and_conditions))
   result['Fun_And'] = f"[{', '.join(and_conditions)}]" if and_conditions else ''
   # Ensure all expected columns exist with empty strings
   for col in ['Fun_Or', 'Fun_And'] + [f'Fun_Or_{i}' for i in range(1, 5)]:
       if col not in result:
           result[col] = ''
   return result
# Apply the function
condition_cols = df['Function'].apply(
   lambda x: pd.Series(extract_conditions(x))
).fillna('')
# Merge with original dataframe
df = pd.concat([df, condition_cols], axis=1)

print('1/3 - PPL Feature Extraction Done...!')

1/3 - PPL Feature Extraction Done...!


In [78]:
#df.to_csv(Path+'Output\\output_new.csv')
#df.head(2)

In [79]:
# Drop columns where all values are empty strings or whitespace
df = df.loc[:, ~(df.apply(lambda col: col.astype(str).str.strip().eq('').all()))]


In [80]:
#df.head(2)

In [81]:
col = ['Function','Class']
df.drop(columns = col, inplace=True)

## Function Extraction

In [82]:
#df = df[['Item','Fun_Or','Fun_And','Condition']]
#df.head(2)

In [83]:
# Function to prefix the first 4 characters of Item to any 'Fun_' column
def prefix_item_to_fun(item, fun):
    if fun and isinstance(fun, str) and fun.strip():
        prefix = item[:4]
        fun_list = fun.strip('[]').split(', ')
        return [f"{prefix}-{f}" for f in fun_list]
    return ''

# Identify all columns that start with 'Fun_'
fun_columns = [col for col in df.columns if col.startswith('Fun_')]

# Apply the function to each 'Fun_' column dynamically
for col in fun_columns:
    df[col] = df.apply(lambda row: prefix_item_to_fun(row['Item'], row[col]), axis=1)

# Optional: Create 'Class' column from first 4 characters of 'Item'
df['Class'] = df['Item'].str[:4]


In [84]:
#df.head()

In [85]:
#data_fin = pd.merge(data_pv, df[['Class','Item', 'Fun_Or','Fun_And','Condition']], on='Class', how='outer')
data_fin = pd.merge(data_pv, df, on='Class', how='outer')

In [86]:
data_fin = data_fin[data_fin['Item'].notna()]

In [87]:
#data_fin.to_csv(Path+'Output\\data_fun.csv',index=False)
#data_fin.head(2)
print('2/3 - PPL Function Extraction Done...!')

2/3 - PPL Function Extraction Done...!


# Pivot Vs Function

# PPL Data Update in Pivot

In [88]:

df1 = data_fin.copy()
df2 = data_pivot.copy()
#df1 = df1[df1['Condition'] != 1]


In [89]:
# Convert df2 to a dictionary for fast lookup
df2_dict = df2.set_index('BOM Parent Product ID').to_dict(orient='index')

# Function to calculate quantities for Fun_Or columns
def calculate_qty_fun(row, fun_col, df2_dict):
    bom_id = row['BOM Parent Product ID']
    if bom_id in df2_dict:
        df_row = df2_dict[bom_id]
        return sum([df_row.get(code, 0) for code in row[fun_col]])
    return 0

# Identify all columns that start with 'Fun_Or'
fun_columns = [col for col in df1.columns if col.startswith('Fun_Or')]

# Apply the function to each 'Fun_Or' column dynamically
for i, col in enumerate(fun_columns):
    new_col_name = f"Fun_Or_Qty_{i}" if i > 0 else "Fun_Or_Qty"
    df1[new_col_name] = df1.apply(lambda row: calculate_qty_fun(row, col, df2_dict), axis=1)



In [90]:
fun_columns_Qty = [col for col in df1.columns if col.startswith('Fun_Or_Q')]
#fun_columns_Qty

In [91]:
## New Code
def calculate_qty_Fun_And(row):
   bom_id = row['BOM Parent Product ID']
   # Check if BOM Parent Product ID matches
   if bom_id in df2['BOM Parent Product ID'].values:
       # Get the row from df2 where BOM Parent Product ID matches
       df_row = df2[df2['BOM Parent Product ID'] == bom_id]
       # Check if all codes in Fun_And exist in df_row columns
       missing_columns = [code for code in row['Fun_And'] if code not in df_row.columns]
       if missing_columns:
           return pd.Series([0])
       # Calculate the minimum quantity
       extracted_codes_qty = min([df_row[code].values[0] for code in row['Fun_And']], default=0)
       return pd.Series([extracted_codes_qty])
   return pd.Series([0])

# Apply the function to each row and create new columns
df1['Fun_And_Qty'] = df1.apply(calculate_qty_Fun_And, axis=1)

In [92]:
############# Get Mini Value

import numpy as np

# Replace inf with a finite number (e.g., 0) before converting to int
df1[fun_columns_Qty] = df1[fun_columns_Qty].replace([np.inf, -np.inf], 0).fillna(0).astype(int)
df1[fun_columns_Qty] = df1[fun_columns_Qty].replace([np.inf, -np.inf], 0).fillna(0).astype(int)

# # Replace inf with a finite number (e.g., 0) before converting to int
# df1['Fun_Or_Qty'] = df1['Fun_Or_Qty'].replace([np.inf, -np.inf], 0).fillna(0).astype(int)
# df1['Fun_And_Qty'] = df1['Fun_And_Qty'].replace([np.inf, -np.inf], 0).fillna(0).astype(int)


#######################################################
# df1.loc[df1['Fun_Or'] == "", 'Fun_Or_Qty'] = np.nan
# df1.loc[df1['Fun_And'] == "", 'Fun_And_Qty'] = np.nan


val_1 = 'Fun_Or'
val_2 = 'Fun_Or_Qty'
co = 0
a = None
for x in fun_columns_Qty:
    if co == 0:
        df1.loc[df1[val_1] == "", val_2] = np.nan
    else:
        a = f"{val_1}_{co}"
        b = f"{val_2}_{co}"
        df1.loc[df1[a] == "", b] = np.nan
    co += 1


df1.loc[df1['Fun_And'] == "", 'Fun_And_Qty'] = np.nan
#######################################################


In [93]:
index = df1.columns.get_loc('Fun_Or_Qty')
#print(df1.columns[8])

In [94]:
# Function to get the minimum value from column index 4 onward
def get_min_value(row):
    values = row[index:]
    if values.isnull().all():
        return ''
    return values.min()

# Apply the function to each row
df1['Qty'] = df1.apply(get_min_value, axis=1)


In [95]:
data = df1.copy()

In [96]:
data.loc[data['Condition']==2, 'Qty'] = 0


In [97]:
data['Qty'] = pd.to_numeric(data['Qty'], errors='coerce')

data['Qty'].fillna(0, inplace=True)


In [98]:
data.to_csv(Path+'Output\\RFC_PPL_Logical.csv',index=False)

In [99]:

pivot_df = data.pivot_table(index='BOM Parent Product ID', columns='Item', values='Qty', fill_value=0)

pivot_df.reset_index(inplace=True)


#pivot_df.to_csv(Path+'Output\\RFC_Transport.csv',index=False)

In [100]:
data_pivot = data_pivot_Bk.copy()

In [101]:
data_pivot = pd.merge(data_pivot, pivot_df,on=['BOM Parent Product ID'],how='left')

In [102]:
data_pivot = data_pivot.fillna(0).round(0)

#data_pivot.to_csv(Path+'Output\\Updated_Pivot.csv',index=False)
#data.head()

In [103]:
print('3/3 - PPL Logic Completed')

3/3 - PPL Logic Completed


In [104]:
#User_Input = input('Enter to Continue : ')

# RFC Logic Initial Sheet

In [105]:
print('RFC Logic Started..!')
data_pivot['Class'] = data_pivot['BOM Parent Product ID'].str[0:4]

RFC Logic Started..!


In [106]:
data = pd.read_excel(Path+'Input\\Logic.xlsx', sheet_name='Logic')
data_logic_bk = data[['Item','Function']].copy()
data_logic_bk.drop_duplicates(subset=['Item'],inplace=True)

#### Replace , and within () to OR

In [107]:

def replace_commas(match):
    return match.group(0).replace(',', ' , ')


def replace_commas_to_or(match):
    return match.group(0).replace(',', ' OR')


data['Function'] = data['Function'].apply(lambda x: re.sub(r'\(.*?\)', replace_commas, x)if isinstance(x, str) else x)
data['Function'] = data['Function'].apply(lambda x: re.sub(r'\(.*?\)', replace_commas_to_or, x)if isinstance(x, str) else x)




In [108]:
data_pv = data_pivot[['Class', 'BOM Parent Product ID']]
data_pv = data_pv.copy()
df = data.copy()
df = df[(df['Function'].notna()) & (df['Function'].str.strip() != '')]
df['Class'] = df['Item'].str[0:4]

df_bk = df.copy()

#### Feature Extraction

In [109]:
import pandas as pd
import re
def extract_conditions(text):
   if pd.isna(text) or text == '':
       return {'Fun_Or': '', 'Fun_And': ''}
   result = {}
   remaining_text = text
   # First process parenthesized OR groups
   or_groups = re.findall(r'\((.*?)\b(?:OR|or)\b(.*?)\)', text, re.IGNORECASE)
   # Process each OR group
   for i, group in enumerate(or_groups):
       or_part = ' '.join([part.strip() for part in group if part.strip()])
       codes = re.findall(r'\b(F\d{3}|OP\d{2}|D\d{3}|PS\d{2}|PS\d{3}|VR\d{2}|VF\d{2})\b', or_part)
       col_name = 'Fun_Or' if i == 0 else f'Fun_Or_{i}'
       result[col_name] = f"[{', '.join(codes)}]" if codes else ''
       remaining_text = remaining_text.replace(f"({or_part})", "", 1)
   # Now process non-parenthesized OR conditions in remaining text
   or_segments = [seg.strip() for seg in remaining_text.split(',')
                 if seg.strip() and re.search(r'\b(?:OR|or)\b', seg, re.IGNORECASE)]
   if or_segments and 'Fun_Or' not in result:
       # Only process if we haven't found any parenthesized OR groups yet
       all_or_codes = []
       for segment in or_segments:
           codes = re.findall(r'\b(F\d{3}|OP\d{2}|D\d{3}|PS\d{2}|PS\d{3}|VR\d{2}|VF\d{2})\b', segment)
           all_or_codes.extend(codes)
       if all_or_codes:
           result['Fun_Or'] = f"[{', '.join(all_or_codes)}]"
   # Process AND conditions in remaining text
   and_conditions = []
   segments = [seg.strip() for seg in remaining_text.split(',') if seg.strip()]
   for segment in segments:
       if not re.search(r'\b(?:OR|or)\b', segment, re.IGNORECASE):
           if code := re.search(r'\b(F\d{3}|OP\d{2}|D\d{3}|PS\d{2}|PS\d{3}|VR\d{2}|VF\d{2})\b', segment):
               and_conditions.append(code.group())
   and_conditions = list(dict.fromkeys(and_conditions))
   result['Fun_And'] = f"[{', '.join(and_conditions)}]" if and_conditions else ''
   # Ensure all expected columns exist with empty strings
   for col in ['Fun_Or', 'Fun_And'] + [f'Fun_Or_{i}' for i in range(1, 5)]:
       if col not in result:
           result[col] = ''
   return result
    
# Apply the function
# condition_cols = df['Function'].apply(
#    lambda x: pd.Series(extract_conditions(x))
# ).fillna('')

condition_cols = df['Function'].apply(
    lambda x: pd.Series(extract_conditions(x)) if isinstance(x, str) else pd.Series()
).fillna('')




# Merge with original dataframe
df = pd.concat([df, condition_cols], axis=1)

In [110]:
#df.to_csv(Path+'Output\\Output.csv',index=False)
print('1/3 - Feature Extraction Done...!')

1/3 - Feature Extraction Done...!


In [111]:
# Drop columns where all values are empty strings or whitespace
df = df.loc[:, ~(df.apply(lambda col: col.astype(str).str.strip().eq('').all()))]

col = ['Function','Class']
df.drop(columns = col, inplace=True)

In [112]:
# Function to prefix the first 4 characters of Item to any 'Fun_' column
def prefix_item_to_fun(item, fun):
    if fun and isinstance(fun, str) and fun.strip():
        prefix = item[:4]
        fun_list = fun.strip('[]').split(', ')
        return [f"{prefix}-{f}" for f in fun_list]
    return ''

# Identify all columns that start with 'Fun_'
fun_columns = [col for col in df.columns if col.startswith('Fun_')]

# Apply the function to each 'Fun_' column dynamically
for col in fun_columns:
    df[col] = df.apply(lambda row: prefix_item_to_fun(row['Item'], row[col]), axis=1)

# Optional: Create 'Class' column from first 4 characters of 'Item'
df['Class'] = df['Item'].str[:4]


In [113]:
data_fin = pd.merge(data_pv, df, on='Class', how='outer')

In [114]:
data_fin = data_fin[data_fin['Item'].notna()]
print('2/3 - Function Extraction Done...!')

2/3 - Function Extraction Done...!


## Pivot Vs Function

#### Logical Opetations [ Condition 1 - ( 1- Sum ) ]

In [115]:

df1 = data_fin.copy()
df2 = data_pivot.copy()


#df1 = df1[df1['Condition']== 1]



In [116]:
# Convert df2 to a dictionary for fast lookup
df2_dict = df2.set_index('BOM Parent Product ID').to_dict(orient='index')

# Function to calculate quantities for Fun_Or columns
def calculate_qty_fun(row, fun_col, df2_dict):
    bom_id = row['BOM Parent Product ID']
    if bom_id in df2_dict:
        df_row = df2_dict[bom_id]
        return sum([df_row.get(code, 0) for code in row[fun_col]])
    return 0

# Identify all columns that start with 'Fun_Or'
fun_columns = [col for col in df1.columns if col.startswith('Fun_Or')]

# Apply the function to each 'Fun_Or' column dynamically
for i, col in enumerate(fun_columns):
    new_col_name = f"Fun_Or_Qty_{i}" if i > 0 else "Fun_Or_Qty"
    df1[new_col_name] = df1.apply(lambda row: calculate_qty_fun(row, col, df2_dict), axis=1)



In [117]:
fun_columns_Qty = [col for col in df1.columns if col.startswith('Fun_Or_Q')]
#fun_columns_Qty

In [118]:
### New Code
def calculate_qty_Fun_And(row):
   bom_id = row['BOM Parent Product ID']
   # Check if BOM Parent Product ID matches
   if bom_id in df2['BOM Parent Product ID'].values:
       # Get the row from df2 where BOM Parent Product ID matches
       df_row = df2[df2['BOM Parent Product ID'] == bom_id]
       # Check if all codes in Fun_And exist in df_row columns
       missing_columns = [code for code in row['Fun_And'] if code not in df_row.columns]
       if missing_columns:
           return pd.Series([0])
       # Calculate the minimum quantity
       extracted_codes_qty = min([df_row[code].values[0] for code in row['Fun_And']], default=0)
       return pd.Series([extracted_codes_qty])
   return pd.Series([0])

# Apply the function to each row and create new columns
df1['Fun_And_Qty'] = df1.apply(calculate_qty_Fun_And, axis=1)

In [119]:
############# Get Mini Value

import numpy as np

# Replace inf with a finite number (e.g., 0) before converting to int
df1[fun_columns_Qty] = df1[fun_columns_Qty].replace([np.inf, -np.inf], 0).fillna(0).astype(int)
df1[fun_columns_Qty] = df1[fun_columns_Qty].replace([np.inf, -np.inf], 0).fillna(0).astype(int)

# # Replace inf with a finite number (e.g., 0) before converting to int
# df1['Fun_Or_Qty'] = df1['Fun_Or_Qty'].replace([np.inf, -np.inf], 0).fillna(0).astype(int)
# df1['Fun_And_Qty'] = df1['Fun_And_Qty'].replace([np.inf, -np.inf], 0).fillna(0).astype(int)


#######################################################
# df1.loc[df1['Fun_Or'] == "", 'Fun_Or_Qty'] = np.nan
# df1.loc[df1['Fun_And'] == "", 'Fun_And_Qty'] = np.nan


val_1 = 'Fun_Or'
val_2 = 'Fun_Or_Qty'
co = 0
a = None
for x in fun_columns_Qty:
    if co == 0:
        df1.loc[df1[val_1] == "", val_2] = np.nan
    else:
        a = f"{val_1}_{co}"
        b = f"{val_2}_{co}"
        df1.loc[df1[a] == "", b] = np.nan
    co += 1


df1.loc[df1['Fun_And'] == "", 'Fun_And_Qty'] = np.nan
#######################################################
index = df1.columns.get_loc('Fun_Or_Qty')
#print(df1.columns[8])

#### Get values from AND conditon - New logic changes

In [120]:
fun_columns = [col for col in df1.columns if col.startswith('Fun_And_Qty')]

def get_and_values(row):
    for x in fun_columns:
        if pd.isnull(row[x]):
            return ''
    return 'Yes'

df1['And_Qty_values'] = df1.apply(get_and_values, axis=1)

In [121]:

#print(df1.columns[df1.shape[1]-2])
fin_index = df1.shape[1]-1


In [122]:
qty_cols = df1.columns[index:fin_index]

def compute_qty(row):
    # Ensure numeric so min/sum behave even if some cells are strings
    values = pd.to_numeric(row[qty_cols], errors='coerce')
    and_flag = str(row.get('And_Qty_values', '')).strip().lower()

    if and_flag == 'yes':
        m = values.min(skipna=True)
        return m
        # If you prefer '' when all are NaN even in the "Yes" case, use:
        # return '' if pd.isna(m) else m

    elif values.isnull().all():
        return ''
    else:
        return values.sum(skipna=True)

df1['Qty'] = df1.apply(compute_qty, axis=1)


In [123]:
df1.drop(columns='And_Qty_values', inplace=True)

###  New Logic

In [124]:

data_logic_bk.loc[
    data_logic_bk['Function'].str.contains(r'\),\(|\), \(|\) ,\(|\) , \(', regex=True),
    'get_Min_Validation'
] = "Yes"

data_logic_bk = data_logic_bk[data_logic_bk['get_Min_Validation']=='Yes']

In [126]:
df1 = pd.merge(df1, data_logic_bk[['Item','get_Min_Validation']], on ='Item', how='left')

In [128]:
#df1.to_csv(Path+'Output\\Before.csv',index=False)

In [129]:
# Ensure numeric conversion for all dynamic columns
df1[fun_columns_Qty] = df1[fun_columns_Qty].apply(pd.to_numeric, errors='coerce')

# Compute minimum across dynamic columns only when data_logic_bk == "Yes"
df1['Qty_Logic'] = np.where(
    df1['get_Min_Validation'] == "Yes",
    df1[fun_columns_Qty].min(axis=1),
    np.nan  # or keep original value if needed
)


In [130]:
df1.loc[df1['get_Min_Validation']=='Yes','Qty'] = df1['Qty_Logic']

In [131]:
#df1.to_csv(Path+'Output\\after.csv',index=False)

In [135]:
df.drop(columns=['get_Min_Validation', 'Qty_Logic'], errors='ignore', inplace=True)

### Exisitng

In [136]:
df1['Qty'] = pd.to_numeric(df1['Qty'], errors='coerce')  # Convert to numeric, set invalid parsing as NaN
df1.loc[df1['Condition'] == 1, 'Qty'] = 1 - df1['Qty']


#### Logical Opetations [ Condition 2 & 3 & 4 & 5 & 6  - Get min values ]

In [137]:
df1.loc[df1['Condition']==2, 'Qty'] = 0

df1['Qty'] = pd.to_numeric(df1['Qty'], errors='coerce')

df1['Qty'].fillna(0, inplace=True)


In [138]:
#df1.to_csv(Path+'Output\\RFC_Logical.csv',index=False)
print('3/3 - RFC Process Completed...!')

3/3 - RFC Process Completed...!


# Final File Preparation

In [139]:
MCID_Break = pd.read_csv(Path+'Input\\MCID Break.csv')

values = {'Inventory Product ID Component':'MCID_Break_Item','Component Qty':'MCID_Break_Qty'}

MCID_Break.rename(columns=values, inplace=True)

C:\Users\vv185114\AppData\Local\Temp\8\ipykernel_27236\989958215.py:1: DtypeWarning: Columns (0,1,3,4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  MCID_Break = pd.read_csv(Path+'Input\\MCID Break.csv')


In [140]:
Logic = pd.read_excel(Path+'Input\\Logic.xlsx', sheet_name='Logic')

In [141]:
data = pd.merge(df1[['Class','BOM Parent Product ID','Item','Condition','Qty']], Logic[['Item','Function']], on='Item', how='left')
data = data[['Class','BOM Parent Product ID','Item','Function','Condition','Qty']]


# MCID Break Vs RFC File

In [142]:
MCID_Break = pd.merge(data,MCID_Break[['MCID_Break_Item','MCID_Break_Qty','BOM Parent Product ID']], left_on=['Item','BOM Parent Product ID'], right_on=['MCID_Break_Item','BOM Parent Product ID'], how='left')



MCID_Break['RFC_vs_MCID'] = MCID_Break['Qty'] - MCID_Break['MCID_Break_Qty'].fillna(0)

MCID_Break.loc[MCID_Break['RFC_vs_MCID'] == 0 , 'Ref'] = 'No Diff'
MCID_Break.loc[MCID_Break['MCID_Break_Item'].isnull(), 'Ref'] = 'New Feature'
MCID_Break.loc[MCID_Break['Ref'].isnull() , 'Ref'] = 'Diff'

# Drop Duplicates

In [143]:
data.drop_duplicates(subset=['BOM Parent Product ID','Item'],inplace=True)
df1.drop_duplicates(subset=['BOM Parent Product ID','Item'],inplace=True)
MCID_Break.drop_duplicates(subset=['BOM Parent Product ID','Item'],inplace=True)

In [144]:
with pd.ExcelWriter(Path+'Output\\RFC.xlsx', engine='xlsxwriter') as writer:
    # Write each DataFrame to a specific sheet
    data.to_excel(writer, sheet_name='RFC', index=False)
    df1.to_excel(writer, sheet_name='Logic_Splitup', index=False)
    MCID_Break.to_excel(writer, sheet_name='RFC_Vs_MCIDBreak', index=False)
    

In [145]:
print('======================================\n.............RFC Completed............\n======================================')


.............RFC Completed............
